01/09/2025

In [40]:
1.65/4

0.4125

In [1]:
pwd

'/home/nampv1/projects/asr/asr_ft/notebooks/postprocessinig'

In [2]:
import sys
sys.path.append("/home/nampv1/projects/asr/asr-demo-app/backend/app/")
sys.path.append("/home/nampv1/projects/asr/asr-demo-app/backend/app/services")
sys.path.append("/home/nampv1/projects/asr/asr-demo-app/backend/app/services/postprocessing")

In [3]:
from postprocess import postprocess_number

/home/nampv1/anaconda3/envs/asr/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
asr_texts = [
    "một ngàn hai trăm năm mươi",
    "một ngàn lẻ bốn",
    "một ngàn không trăm lẻ bốn",
    "một trăm linh bảy",
    "một trăm ba mươi hai",
    "một trăm",
    "hai trăm ninh hai ngàn ba trăm hai mưoi mốt",
    "năm mưoi triệu hai trăm ngàn"
]

for text in asr_texts:
    output = postprocess_number(text)
    print(output)

một ngàn 250
một ngàn lẻ bốn
một ngàn 4
107
132
một trăm
202 ngàn 302 mưoi mốt
năm mưoi 1000200 ngàn


In [41]:
import random

def generate_multi_number_sequence_test(single_number_dataset, max_numbers=3, seed=None):
    """
    Tạo các câu gồm nhiều số đọc liên tiếp.
    
    Args:
        single_number_dataset: list[(number, vietnamese_text)]
        max_numbers: số lượng số nối liền tối đa
        seed: để reproducible
    Returns:
        list[(sentence, expected_normalized)]
    """
    if seed is not None:
        random.seed(seed)
        
    results = []
    
    # số lượng thử mỗi case
    num_cases = 20
    
    for _ in range(num_cases):
        n_numbers = random.randint(2, max_numbers)
        selected = random.sample(single_number_dataset, n_numbers)
        
        # nối text để tạo sentence
        sentence = " ".join(vn for _, vn in selected)
        expected = " ".join(str(n) for n, _ in selected)
        
        results.append((sentence, expected))
        
    return results


# --- Demo ---
if __name__ == "__main__":
    # dataset đơn: [(number, vn_text)]
    single_number_dataset = generate_test_dataset(max_digits=3)  # từ 1–3 chữ số
    
    multi_number_tests = generate_multi_number_sequence_test(single_number_dataset, max_numbers=3, seed=42)
    
    for sentence, expected in multi_number_tests:
        print(f"Text: {sentence}")
        print(f"Expected: {expected}")
        print("-"*50)


Text: ba một trăm lẻ hai
Expected: 3 102
--------------------------------------------------
Text: ba mươi một trăm tám mươi chín một trăm lẻ hai
Expected: 30 189 102
--------------------------------------------------
Text: một trăm lẻ hai một trăm tám mươi chín
Expected: 102 189
--------------------------------------------------
Text: một trăm bảy mươi một trăm
Expected: 170 100
--------------------------------------------------
Text: ba một trăm tám mươi chín
Expected: 3 189
--------------------------------------------------
Text: ba mươi một trăm bảy mươi
Expected: 30 170
--------------------------------------------------
Text: một trăm bảy mươi ba mươi
Expected: 170 30
--------------------------------------------------
Text: ba mươi một trăm một trăm bảy mươi
Expected: 30 100 170
--------------------------------------------------
Text: một trăm tám mươi chín ba ba mươi
Expected: 189 3 30
--------------------------------------------------
Text: ba mươi lăm một trăm tám mươi chín ba m

In [8]:
import os

path = "/home/nampv1/projects/asr/asr-demo-app/backend/tests/test_postprocess.py"
start = "/home/nampv1/projects/asr/asr-demo-app/backend/app/services/postprocessing/"

rel_path = os.path.relpath(start, path)
print(rel_path)  # myapp/file.txt


../../app/services/postprocessing


In [1]:
pwd

'/home/nampv1/projects/asr/asr_ft/notebooks/postprocessinig'

In [22]:
def number_to_vietnamese(n, zero_read="lẻ"):
    units = ["", "một", "hai", "ba", "bốn", "năm", "sáu", "bảy", "tám", "chín"]
    tens_words = ["", "mười", "hai mươi", "ba mươi", "bốn mươi", "năm mươi",
                  "sáu mươi", "bảy mươi", "tám mươi", "chín mươi"]
    scales = ["", "nghìn", "triệu", "tỷ", "nghìn tỷ", "triệu tỷ", "tỷ tỷ"]

    def read_three_digits(num, is_first_group=False):
        hundred = num // 100
        ten = (num % 100) // 10
        one = num % 10
        result = []

        # Hàng trăm
        if hundred > 0:
            result.append(units[hundred] + " trăm")
        elif not is_first_group and num != 0:
            result.append("không trăm")

        # Hàng chục
        if ten > 1:
            result.append(tens_words[ten])
        elif ten == 1:
            result.append("mười")
        elif ten == 0 and one > 0 and (hundred > 0 or not is_first_group):
            result.append(zero_read)

        # Hàng đơn vị
        if one > 0:
            if ten == 0 or ten == 1:
                if one == 5 and ten > 0:
                    result.append("lăm")
                else:
                    result.append(units[one])
            else:
                if one == 1:
                    result.append("mốt")
                elif one == 4:
                    result.append("tư")
                elif one == 5:
                    result.append("lăm")
                else:
                    result.append(units[one])

        return " ".join(result)

    if n == 0:
        return "không"

    # Chia số thành các nhóm 3 chữ số
    str_n = str(n)
    groups = []
    while str_n:
        groups.insert(0, int(str_n[-3:]))
        str_n = str_n[:-3]

    words = []
    group_len = len(groups)
    for idx, g in enumerate(groups):
        is_first_group = (idx == 0)
        # Nếu nhóm ≠ 0 hoặc nằm giữa các nhóm còn số khác thì mới đọc
        has_nonzero_after = any(groups[idx + 1:]) if idx + 1 < group_len else False
        if g != 0 or has_nonzero_after:
            group_words = read_three_digits(g, is_first_group)
            if group_words:
                words.append(group_words)
            scale = scales[group_len - idx - 1]
            if scale and (g != 0 or has_nonzero_after):
                words.append(scale)

    return " ".join(words).strip()


# Test các case bạn vừa nêu
examples = [304, 102104, 101114001, 2100010, 1002003, 101, 1205, 2345678901]
examples = [1000000005, 1005, 1000004]
for num in examples:
    print(f"{num}: {number_to_vietnamese(num, zero_read='lẻ')}")
    print(f"{num}: {number_to_vietnamese(num, zero_read='linh')}\n")


1000000005: một tỷ triệu nghìn không trăm lẻ năm
1000000005: một tỷ triệu nghìn không trăm linh năm

1005: một nghìn không trăm lẻ năm
1005: một nghìn không trăm linh năm

1000004: một triệu nghìn không trăm lẻ bốn
1000004: một triệu nghìn không trăm linh bốn



In [ ]:
examples = [
    2000
    200004,
    200000,
    10226,
    10015,
    10007,
    10000,
    50200000,
    200
]

for num in examples:
    print(f"{num}: {number_to_vietnamese(num, zero_read='lẻ')}")

200004: hai trăm nghìn không trăm lẻ bốn
200000: hai trăm nghìn
10226: mười nghìn hai trăm hai mươi sáu
10015: mười nghìn không trăm mười lăm
10007: mười nghìn không trăm lẻ bảy
10000: mười nghìn
50200000: năm mươi triệu hai trăm nghìn
200: hai trăm


In [38]:
import random

def generate_test_cases(d, samples=None):
    """
    Sinh số đại diện cho số có d chữ số theo quy luật 2^(d-1).
    samples: list các chữ số ≠0 để random, mặc định [1..9]
    """
    if samples is None:
        samples = list(range(1, 10))

    results = []
    first_digit = random.choice(samples)  # chữ số đầu luôn ≠0

    # Sinh tất cả 2^(d-1) pattern cho các chữ số còn lại
    for mask in range(2**(d-1)):
        digits = [first_digit]
        for pos in range(d-1):
            if (mask >> pos) & 1:
                digits.append(random.choice(samples))  # ≠0
            else:
                digits.append(0)
        n = int("".join(str(x) for x in digits))
        results.append(n)

    return results


def generate_test_dataset(max_digits=6):
    dataset = []
    for d in range(1, max_digits+1):
        numbers = generate_test_cases(d)
        for n in numbers:
            dataset.append((n, number_to_vietnamese(n)))
    return dataset


# Demo
if __name__ == "__main__":
    dataset = generate_test_dataset(max_digits=6)
    for n, vn in dataset:
        print(f"{n} -> {vn}")


8 -> tám
50 -> năm mươi
51 -> năm mươi mốt
200 -> hai trăm
230 -> hai trăm ba mươi
205 -> hai trăm lẻ năm
288 -> hai trăm tám mươi tám
7000 -> bảy nghìn
7300 -> bảy nghìn ba trăm
7090 -> bảy nghìn không trăm chín mươi
7940 -> bảy nghìn chín trăm bốn mươi
7004 -> bảy nghìn không trăm lẻ bốn
7603 -> bảy nghìn sáu trăm lẻ ba
7097 -> bảy nghìn không trăm chín mươi bảy
7356 -> bảy nghìn ba trăm năm mươi sáu
60000 -> sáu mươi nghìn
67000 -> sáu mươi bảy nghìn
60300 -> sáu mươi nghìn ba trăm
67500 -> sáu mươi bảy nghìn năm trăm
60030 -> sáu mươi nghìn không trăm ba mươi
64020 -> sáu mươi tư nghìn không trăm hai mươi
60530 -> sáu mươi nghìn năm trăm ba mươi
63780 -> sáu mươi ba nghìn bảy trăm tám mươi
60004 -> sáu mươi nghìn không trăm lẻ bốn
61004 -> sáu mươi mốt nghìn không trăm lẻ bốn
60602 -> sáu mươi nghìn sáu trăm lẻ hai
62609 -> sáu mươi hai nghìn sáu trăm lẻ chín
60042 -> sáu mươi nghìn không trăm bốn mươi hai
66095 -> sáu mươi sáu nghìn không trăm chín mươi lăm
60289 -> sáu mươi nghìn

---

In [ ]:
import re

VI_NUM_MAP = {
    "không": 0, "một": 1, "mốt": 1, "hai": 2, "ba": 3,
    "bốn": 4, "tư": 4, "năm": 5, "lăm": 5, "sáu": 6,
    "bảy": 7, "tám": 8, "chín": 9
}

MAJOR_SCALES = {"tỷ", "triệu", "nghìn", "ngàn"}
HIGH_MAJOR = {"tỷ", "triệu"}        # scale lớn cần thận trọng hơn
TENS_MARKERS = {"mươi", "mười"}
HUNDRED = "trăm"
LE_MARKERS = {"lẻ", "linh"}

def _is_unit(tok): return tok in VI_NUM_MAP
def _contains_any(tokens, set_words): return any(t in set_words for t in tokens)

def validate_lower_group(group_tokens):
    """
    Validate a 0..999 lower-group is explicit:
    - 'lẻ'/'linh' => explicit
    - tens marker => explicit
    - contains 'trăm':
        * nothing after -> explicit (e.g. 'sáu trăm' -> 600)
        * >=2 tokens after -> explicit ('sáu trăm hai mươi' -> explicit)
        * exactly 1 token after and it's a bare unit -> AMBIGUOUS -> NOT explicit
    - no 'trăm'/'mươi'/'lẻ': single bare unit -> NOT explicit
    - otherwise -> explicit
    """
    if not group_tokens: return False
    if _contains_any(group_tokens, LE_MARKERS): return True
    if _contains_any(group_tokens, TENS_MARKERS): return True

    if HUNDRED in group_tokens:
        idx = group_tokens.index(HUNDRED)
        after = group_tokens[idx + 1:]
        if len(after) == 0:
            return True
        if len(after) >= 2:
            return True
        if len(after) == 1 and _is_unit(after[0]):
            return False
        return False

    if len(group_tokens) == 1 and _is_unit(group_tokens[0]):
        return False

    return True

def normalize_number_sequence_strict(span_text, vietnamese_to_number_fn):
    """
    Strict normalizer: convert only when spoken form is explicit enough.
    vietnamese_to_number_fn: callable(str) -> int or None
    """
    if not span_text or not span_text.strip():
        return span_text
    s = span_text.lower().strip()

    # pure digits -> join
    if re.fullmatch(r"(?:\d+\s*)+", s):
        return "".join(s.split())

    tokens = s.split()

    # if explicit le/linh anywhere -> allow
    if _contains_any(tokens, LE_MARKERS):
        n = vietnamese_to_number_fn(s)
        return str(n) if n is not None else span_text

    # find major scale indices
    major_idx = [i for i, t in enumerate(tokens) if t in MAJOR_SCALES]

    # no major scale -> validate whole phrase as a lower group
    if not major_idx:
        if validate_lower_group(tokens):
            n = vietnamese_to_number_fn(s)
            return str(n) if n is not None else span_text
        else:
            return span_text

    # validate multipliers before each major scale
    prev = 0
    for k, idx in enumerate(major_idx):
        multiplier_tokens = tokens[prev:idx]
        if not multiplier_tokens:
            return span_text  # e.g., starts with scale -> ambiguous

        mul_txt = " ".join(multiplier_tokens)

        if k == 0:
            # left-most major scale: allow single bare unit multiplier (e.g., "một tỷ")
            mul_val = vietnamese_to_number_fn(mul_txt)
            if mul_val is None: return span_text
        else:
            # subsequent multipliers must be explicit groups
            if not validate_lower_group(multiplier_tokens):
                return span_text
            mul_val = vietnamese_to_number_fn(mul_txt)
            if mul_val is None: return span_text

        prev = idx + 1

    # after last major scale, the remainder is the final lower group
    last_group = tokens[prev:]
    if not last_group:
        return span_text

    # STRICT GUARD: if there exists any HIGH_MAJOR (triệu/tỷ) anywhere in tokens,
    # and the final lower-group value is < 100 and final group lacks 'trăm'/'lẻ' -> ambiguous -> keep
    contains_high_major = any(t in HIGH_MAJOR for t in tokens)
    if contains_high_major:
        # compute numeric value of last_group safely (without normalizing whole phrase)
        last_txt = " ".join(last_group)
        last_val = vietnamese_to_number_fn(last_txt)
        lacks_hundred_or_le = (not _contains_any(last_group, {HUNDRED}) and not _contains_any(last_group, LE_MARKERS))
        if last_val is not None and last_val < 100 and lacks_hundred_or_le:
            return span_text

    # otherwise validate last lower group normally
    if not validate_lower_group(last_group):
        return span_text

    # all checks passed -> normalize whole phrase
    n = vietnamese_to_number_fn(s)
    return str(n) if n is not None else span_text


In [5]:
texts = [
    "Hai Trăm Linh Năm Hai Trăm Linh Bảy",
    "hai trăm ninh năm",
    "hai trăm lẻ ba",
    # "một tỷ sáu mươi hai",
    # "một tỷ sáu trăm triệu",
    # "một tỷ sáu",
    # "số một trăm sáu mươi tám",
    # "một triệu sáu nghìn",
    # "sáu trăm bảy",
    # "hai mươi",
    # "một ngàn sáu trăm linh chín",
    # "một nghìn sáu trăm hai mươi hai"
    # "một nghìn không trăm hai mươi ba",
    # "một ngàn sáu",
    # "một trăm hai",
    # "một trăm lẻ hai",
    # "Một trăm linh hai",
    "một ngàn sáu trăm",
    # "số một trăm sáu mươi tám",
    # "sáu trăm linh hai",
    # "sáu trăm hai mươi hai",
    # "một trăm chín mươi nghìn năm trăm",
]

normalize_number_sequence(texts[0])

NameError: name 'normalize_number_sequence' is not defined

In [1]:
# Thêm vào code của bạn
import re

UNITS = {"không","một","mốt","hai","ba","bốn","tư","năm","lăm","sáu","bảy","tám","chín"}
TENS_UNITS = {"mươi","mười"}
HUNDRED = "trăm"
SCALE = {"nghìn","ngàn","triệu","tỷ"}
LITTLE = {"lẻ","linh","ninh","nẻ"}

def split_number_phrases(text):
    tokens = text.lower().strip().split()
    groups = []
    cur = []
    L = len(tokens)
    for i, t in enumerate(tokens):
        cur.append(t)
        # boundary trên từ scale rõ ràng
        if t in SCALE:
            groups.append(" ".join(cur).strip())
            cur = []
            continue
        # nếu nhóm hiện tại đã có trăm/mươi/mười hoặc lẻ/linh -> khả năng nhóm hoàn tất
        has_hundred = HUNDRED in cur
        has_tens = any(x in cur for x in TENS_UNITS)
        has_little = any(x in cur for x in LITTLE)
        if has_hundred or has_tens or has_little:
            if i+1 < L:
                nxt = tokens[i+1]
                # nếu next là chữ số và token kế nữa là trăm/mươi/mười -> chắc chắn bắt đầu nhóm mới
                if nxt in UNITS and i+2 < L and tokens[i+2] in (HUNDRED, *TENS_UNITS):
                    groups.append(" ".join(cur).strip())
                    cur = []
                    continue
                # nếu next là số/chữ và current đã có trăm/mươi -> khả năng split,
                # nhưng không split nếu current kết thúc bằng "lẻ/linh" (thường là nối)
                if (re.fullmatch(r"\d+", nxt) or nxt in UNITS) and (has_hundred or has_tens):
                    if not has_little:
                        groups.append(" ".join(cur).strip())
                        cur = []
                        continue
    if cur:
        groups.append(" ".join(cur).strip())
    return groups


In [2]:
examples = [
    "một triệu sáu trăm hai triệu ba trăm ngàn",
    "hai trăm linh ba bốn trăm lẻ chín",        # 203, 409
    "một triệu hai trăm ba mươi tư nghìn năm trăm",  # 1_234_500
    "một trăm hai mươi ba",                      # 123
    "năm mươi bốn nghìn sáu trăm bảy mươi tám", # 54_678
    "một tỷ hai trăm ba mươi triệu bốn trăm linh năm", # 1_230_405
    "mười một",                                  # 11
    "hai mươi lăm",                              # 25
    "bảy trăm linh chín",                        # 709
    "một nghìn không trăm lẻ năm",               # 1005
    "hai triệu ba trăm nghìn bốn trăm hai mươi", # 2_300_420
]

for ex in examples:
    phrases = split_number_phrases(ex)
    numbers = [vietnamese_to_number(p) for p in phrases]
    print(f"{ex}\n-> Phrases: {phrases}\n-> Numbers: {numbers}\n")
    break


NameError: name 'vietnamese_to_number' is not defined